In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from utils.utils import ComputeROIs
from drift.drift import phase_correlation, pad_data, get_drift_xy, make_template



Operating system:  Linux


Autosaving every 180 seconds


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = '/home/cat/code/bscope_tests/8499/data/Image_001_001.raw'

#fname = r'D:\bmi\DON8498\22-06-08\calibration\Image_001_001.raw'

# 
gr = ComputeROIs(fname)


#
gr.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


In [ ]:
####################################################################
########################## FIND TEMPLATE ###########################
####################################################################

print ("data: ", data.shape)

# 
n_imgs_to_sample = 500
n_best_imgs = 100
corr_maxs, template, idx_imgs = make_template(data, 
                                              n_imgs_to_sample,
                                              n_best_imgs)

# note: output template shoudl be in uint16 to match live image


In [ ]:
####################################################################
##################### FIND DRIFT AND CORRECT DATA ##################
####################################################################

print ("TODO: Implement 8-core parallel version of this...")





# note: show drift during live image



### GAUSIAN SMOOTHIN STEP TO REMOVE PHOTON SHOT NOISE IS THE SLOWEST 
### TODO: try to speed up to go faster plus use more data
###   - NOTE: The more data we use here, the better the cell detection etc.

In [3]:
#######################################################################
########### COMPUTE STD OVER TIME TO GET CELL FOOTPRINTS ##############
#######################################################################
# TODO: the gaussian filter step takes a long time
#      - try to speed up
#      - also, corelation map might be even better method for detecting ROIs
#         e.g. see suite2p's visualization tool <- seems pretty quick

# Also, need to exponse some more variables here like imshow vmin/vmax which are used to plot the final result
start = time.time()

std_map = gr.make_std_map()
print ("total processing time: ", time.time()-start, " sec")
print ("DONE")

memmap :  (10000, 512, 512)
data into analysis:  (1000, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
total processing time:  6.214060306549072  sec
DONE


In [4]:
#################################################################################
########### FIND CORRECT VMIN VMAX FOR MAXIMUM WATERSHED DETECTION ##############
#################################################################################
gr.vmin = 220
gr.vmax = 280

# get a thrsholded map which hides most of the noise
std_map2 = gr.plot_std_map(std_map)
print ("DONE")

staring computing std...
done computing std...
DONE


In [8]:
#######################################################################
########### RUN WATERSHED ALGORITHM DETECTION ON STD MAP ##############
#######################################################################

gr.min_size_roi = 25
gr.max_size_roi = 500
gr.sigma = 0.1
gr.n_smooth_steps = 1
gr.find_roi_boundaries(std_map2)

#
gr.scale=3000                      # spacing between each neuron trace because they are not normalized to the max vlaue
gr.trace_subsample = 10             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2(std_map2)

plotting cells:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
memmap :  (10000, 512, 512)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 6480.79it/s]


[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]


In [9]:
###############################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES #############
###############################################################

# save ensemble rois
gr.ensemble1 = [0,2]
gr.ensemble2 = [6,12]
both = np.hstack((gr.ensemble1, gr.ensemble2))
print ("all cells:", both)

#
gr.show_traces_ids(both)


all cells: [ 0  2  6 12]


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [10]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
gr.trace_subsample = 1             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2(std_map2, both)

print ("DONE...")

plotting cells:  [ 0  2  6 12]
memmap :  (10000, 512, 512)


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:07<00:00, 1425.21it/s]

[ 0  2  6 12]
DONE...


In [17]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
gr.sample_rate = 30
gr.post_reward_lockout = 10   # reward lockout in seconds
gr.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards
gr.rois_smooth_window = 10     # of frames to use to smooth the realtime signal
gr.smooth_diff_function_flag = True    # use a kernel window to smooth current value

# find 30% reward threshold
# We have 3 options: 
#   1) rewarding  E1-E2 going above some threshold
#   2) rewarding E2-E1 going above some threhosld
#   3) rewarding both
# The challenge is mapping the ensemble states to tone/stimulus space
# 
#gr.find_reward_thresholds_low_and_high()
#gr.find_reward_thresholds_high()  # this only rewards when sound passes specific level

# this option rewards both ensembles 
normalize_peaks = True   # this flag normalizes the peaks to make sure one ensembel
                         # doesn't completely dominate the other
gr.find_reward_thresholds_absolute(normalize_peaks)


#
print ("thresholds: ", gr.high)

########################################
gr.plot_rewarded_ensembles()

print (gr.high)
gr.high = gr.high*1

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 9990/9990 [00:00<00:00, 240855.20it/s]


TODO: Normalize the peaks of the 2 Ensembles so the mice don't learn to use aginst the other!!!!
low, high:  nan 1.5063680403707884
nsec recording:  333 max # of random rewards (i.e. every 30sec)  11
updated rwards #:  11 nan 0.9493875682717056
thresholds:  0.9493875682717056
0.9493875682717056


In [14]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# save all data to disk
# also add the tone values here as well that will be used for the experiment
gr.low_freq = 2000
gr.high_freq = 18000

# save cell pixel footprints locations as 2 column array inside list
cells = []
for k in range(4):
    temp = gr.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))
    cells.append(temp)

# also grab contours of cells; both contains all cell ids
contours = gr.compute_contour_map(std_map, both)
print ("len: ", len(contours))        

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(os.path.split(fname)[0])[0],
                        'rois_pixels_and_thresholds.npz'),
            cell0_footprint = cells[0],
            cell1_footprint = cells[1],
            cell2_footprint = cells[2],
            cell3_footprint = cells[3],
            
            #
            cell0_contour = contours[0],
            cell1_contour = contours[1],
            cell2_contour = contours[2],
            cell3_contour = contours[3],
         
            #
            cell_f0s = gr.roi_f0s,
            cell_centres = np.int32(gr.rois)[both],
            cell_ids = both,
            all_rois = np.int32(gr.rois),
            low_threshold = gr.low,
            high_threshold = gr.high,
            low_freq = gr.low_freq,
            high_freq = gr.high_freq,
            cell_traces = gr.roi_traces,
            sample_rate = gr.sample_rate,
            post_reward_lockout = gr.post_reward_lockout,
            balance_ensemble_rewards_flag = gr.balance_ensemble_rewards_flag,
            rois_smooth_window = gr.rois_smooth_window,
            smooth_diff_function_flag = gr.smooth_diff_function_flag
         
        )



len:  4


In [16]:
data = np.load('/home/cat/code/bscope_tests/8499/databmi_results.npz')
ensembles = data['ensemble_activity']

print (ensembles.shape)
plt.figure()
plt.plot(ensembles[0])
plt.plot(ensembles[1])
plt.show()

(2, 10000)


In [35]:
#
print (gr.diff.shape)
plt.figure(0)
plt.plot(gr.diff)
plt.show()

(20000,)


In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)



In [2]:
data = np.load('/media/cat/4TB/donato/BSCOPE_tests/rois_pixels_and_thresholds.npz')

low_thresh = data['low_threshold']
high_thresh = data['high_threshold']

print (low_thresh, high_thresh)



-768.7697339352011 546.5230278373468


In [10]:

octave_size = 0.25

# 
def get_octave_frequencies(low_freq,
                  high_freq,
                  octave_size):
    
    #
    octaves = []
    
    #
    octaves.append(low_freq)
    temp = low_freq
    while True:
        temp = temp * (1+octave_size)
        if temp>high_freq:
            break
        octaves.append(temp)

    return octaves
#
octaves = get_octave_frequencies(2000,
              18000,
              octave_size)
print (len(octaves),octaves)
      
    


10 [2000, 2500.0, 3125.0, 3906.25, 4882.8125, 6103.515625, 7629.39453125, 9536.7431640625, 11920.928955078125, 14901.161193847656]
